# ADAS-Vision — Milestone 4: Huấn luyện & Đánh giá DeepLabV3+ trên BDD100K

Notebook này hướng dẫn quy trình huấn luyện mô hình **DeepLabV3+** (backbone ResNet34) cho bài toán phân vùng làn đường và vạch kẻ đường (3 kênh đa nhãn độc lập) trên tập dữ liệu **BDD100K** sử dụng **Google Colab (GPU T4)**.

### Các kênh phân vùng (SEG_CHANNELS):
0. `drivable_direct` (Làn xe đang lưu thông trực tiếp)
1. `drivable_alternative` (Làn xe lân cận có thể chuyển sang)
2. `lane_marking` (Vạch kẻ sơn đường)

## 1. Kiểm tra phần cứng GPU & Môi trường

In [ ]:
!nvidia-smi

import torch

print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

## 2. Kết nối Google Drive & Cài đặt Thư viện

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

# Clone repository hoặc chuyển vào thư mục code đã mount trên Drive
# %cd /content/drive/MyDrive/Developing-utility-vehicles-and-regional-transit-networks

!pip install --upgrade segmentation-models-pytorch albumentations pyyaml opencv-python-headless

## 3. Khởi tạo Mô hình DeepLabV3+ & Combo Loss

In [ ]:
import segmentation_models_pytorch as smp
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = smp.DeepLabV3Plus(
    encoder_name='resnet34',
    encoder_weights='imagenet',
    in_channels=3,
    classes=3
).to(device)

print("Mô hình DeepLabV3+ đã được khởi tạo thành công trên:", device)

## 4. Huấn luyện Mô hình với scripts/train_segmenter.py

In [ ]:
# Huấn luyện mô hình sử dụng script chuẩn hoá của dự án
!python scripts/train_segmenter.py --epochs 30 --batch-size 8 --device cuda

## 5. Đánh giá Định lượng & Đo lường IoU 3 Kênh

In [ ]:
from pathlib import Path

import torch

checkpoint_path = Path('weights/segmenter_best.pt')
if checkpoint_path.exists():
    ckpt = torch.load(checkpoint_path, map_location='cpu')
    print("=== Kết quả Checkpoint Tốt Nhất ===")
    print("Epoch:", ckpt.get('epoch'))
    for k, v in ckpt.get('metrics', {}).items():
        print(f"{k}: {v:.4f}")
    print("Ngưỡng tối ưu:", ckpt.get('thresholds'))
else:
    print("Chưa có checkpoint weights/segmenter_best.pt!")

## 6. Trực quan hóa Mặt Nạ Phân Vùng Làn Đường

In [ ]:
# from models.segmenter import DeepLabV3PlusSegmenter
# import cv2
# import matplotlib.pyplot as plt
# segmenter = DeepLabV3PlusSegmenter(weights_path='weights/segmenter_best.pt')
# sample_bgr = cv2.imread('data/bdd100k/images/100k/val/cabc30fc-e7726578.jpg')
# if sample_bgr is not None:
#     masks = segmenter.predict(sample_bgr, return_prob=False)
#     plt.figure(figsize=(15, 5))
#     plt.subplot(1, 3, 1); plt.imshow(masks[0], cmap='Greens'); plt.title('Drivable Direct')
#     plt.subplot(1, 3, 2); plt.imshow(masks[1], cmap='Blues'); plt.title('Drivable Alternative')
#     plt.subplot(1, 3, 3); plt.imshow(masks[2], cmap='Reds'); plt.title('Lane Markings')
#     plt.show()